# 02 — Cross-Asset Momentum and Risk Screener

## Why this project exists

Investment teams often need a quick answer to: **"Where is strength concentrated, and where is risk rising?"**

This notebook builds a small ETF screener using Yahoo Finance data. It ranks assets by momentum, volatility and drawdown. The goal is to show that I can go from raw market data to a clean decision-support table.

In [ ]:
!pip install yfinance plotly -q

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.express as px

pd.options.display.float_format = "{:,.4f}".format

## 1. Define the universe

I use liquid ETFs across equities, bonds, commodities and currency proxies. This gives a cross-asset view without needing paid data.

In [ ]:
universe = {
    "SPY": "US equities",
    "QQQ": "US growth / Nasdaq",
    "IWM": "US small caps",
    "EFA": "Developed ex-US equities",
    "EEM": "Emerging markets",
    "TLT": "Long-term US Treasuries",
    "IEF": "Intermediate US Treasuries",
    "GLD": "Gold",
    "SLV": "Silver",
    "DBC": "Commodities",
    "UUP": "US Dollar"
}

tickers = list(universe.keys())
prices = yf.download(tickers, start="2018-01-01", auto_adjust=True, progress=False)["Close"].dropna(how="all")
prices.tail()

## 2. Build features

For a simple investment screener, I want features that answer three questions:

1. **Momentum**: what has been going up?
2. **Risk**: how volatile is it?
3. **Pain**: how far is it from its previous high?

I avoid overengineering at this stage. Simple features are easier to explain in an interview.

In [ ]:
returns = prices.pct_change()

screener = pd.DataFrame(index=tickers)
screener["asset_class"] = pd.Series(universe)
screener["return_1m"] = prices.pct_change(21).iloc[-1]
screener["return_3m"] = prices.pct_change(63).iloc[-1]
screener["return_6m"] = prices.pct_change(126).iloc[-1]
screener["return_12m"] = prices.pct_change(252).iloc[-1]
screener["vol_3m_ann"] = returns.rolling(63).std().iloc[-1] * np.sqrt(252)
screener["drawdown"] = (prices / prices.cummax() - 1).iloc[-1]

# A simple composite momentum score
for col in ["return_1m", "return_3m", "return_6m", "return_12m"]:
    screener[col + "_rank"] = screener[col].rank(ascending=False)

screener["momentum_score"] = screener[["return_1m_rank", "return_3m_rank", "return_6m_rank", "return_12m_rank"]].mean(axis=1)
screener = screener.sort_values("momentum_score")
screener

## 3. Visualize the ranking

A table is useful, but a visual makes it easier to spot whether leadership is concentrated in one asset class or spread across markets.

In [ ]:
plot_df = screener.reset_index().rename(columns={"index": "ticker"})
fig = px.scatter(
    plot_df,
    x="return_6m",
    y="vol_3m_ann",
    size="return_12m",
    color="asset_class",
    hover_name="ticker",
    title="Cross-asset momentum vs risk"
)
fig.show()

px.bar(plot_df.sort_values("momentum_score", ascending=False), x="ticker", y="momentum_score", color="asset_class", title="Lower score = stronger momentum ranking").show()

## 4. My conclusion

This screen is not a trading system. It is a first layer of market intelligence.

The useful output for a trader or portfolio manager is:

- Which assets are leading?
- Which assets have momentum but high volatility?
- Which assets are still in deep drawdown?
- Where should we investigate further?

## Possible next iterations

- Add risk-adjusted momentum
- Add sector ETFs
- Add rolling correlations
- Add a Streamlit interface with selectable universe